# 1. Setup & Environment Configuration

In [ ]:
!pip install -q pandas pyarrow tensorflow "numpy<2" kaggle

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

###  Kaggle API Credentials Setup

To use the Kaggle API, you must authenticate with your account credentials:

1. Go to your **Kaggle Account Settings** and click **Create New Token** to download your `kaggle.json` file.
2. Place the downloaded `kaggle.json` file inside the `~/.kaggle/` directory on your system.


In [ ]:
!kaggle competitions download -c asl-signs

# 2. Exploratory Data Analysis & Landmark Inspection

In [5]:
import pandas as pd

# Load the master label file into a DataFrame
df = pd.read_csv('../data/raw/train.csv')

# Display the first 5 rows to see the structure
df.head()

,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie


In [6]:
# Grab the file path from the very first row of your CSV
first_file_path = df['path'][0]

print(f"Loading data from: {first_file_path}\n")

# Load the coordinate data into a new dataframe
# (PyArrow, which we installed earlier, allows Pandas to read .parquet files)
parquet_df = pd.read_parquet('../data/raw/' + first_file_path)

# Show the first 10 rows of data
parquet_df.head(10)

Loading data from: train_landmark_files/26734/1000035562.parquet



,frame,row_id,type,landmark_index,x,y,z
0,20,20-face-0,face,0,0.494400,0.380470,-0.030626
1,20,20-face-1,face,1,0.496017,0.350735,-0.057565
2,20,20-face-2,face,2,0.500818,0.359343,-0.030283
3,20,20-face-3,face,3,0.489788,0.321780,-0.040622
4,20,20-face-4,face,4,0.495304,0.341821,-0.061152
5,20,20-face-5,face,5,0.496499,0.330019,-0.056744
6,20,20-face-6,face,6,0.501039,0.301457,-0.028559
7,20,20-face-7,face,7,0.436136,0.306364,0.032783
8,20,20-face-8,face,8,0.501792,0.282995,-0.021999
9,20,20-face-9,face,9,0.501045,0.271745,-0.024336


In [7]:
# Check all the unique body parts tracked in this file
print("Original tracking types:", parquet_df['type'].unique())

# Filter the dataframe to ONLY keep the left and right hands
hands_df = parquet_df[parquet_df['type'].isin(['left_hand', 'right_hand'])]

# Display the filtered data
hands_df.head(10)

Original tracking types: ['face' 'left_hand' 'pose' 'right_hand']


,frame,row_id,type,landmark_index,x,y,z
468,20,20-left_hand-0,left_hand,0,NaN,NaN,NaN
469,20,20-left_hand-1,left_hand,1,NaN,NaN,NaN
470,20,20-left_hand-2,left_hand,2,NaN,NaN,NaN
471,20,20-left_hand-3,left_hand,3,NaN,NaN,NaN
472,20,20-left_hand-4,left_hand,4,NaN,NaN,NaN
473,20,20-left_hand-5,left_hand,5,NaN,NaN,NaN
474,20,20-left_hand-6,left_hand,6,NaN,NaN,NaN
475,20,20-left_hand-7,left_hand,7,NaN,NaN,NaN
476,20,20-left_hand-8,left_hand,8,NaN,NaN,NaN
477,20,20-left_hand-9,left_hand,9,NaN,NaN,NaN


In [8]:
# Replace all NaN values with 0.0
clean_hands_df = hands_df.fillna(0)

# Display the same left hand frame to verify the fix
clean_hands_df.head(10)

,frame,row_id,type,landmark_index,x,y,z
468,20,20-left_hand-0,left_hand,0,0.0,0.0,0.0
469,20,20-left_hand-1,left_hand,1,0.0,0.0,0.0
470,20,20-left_hand-2,left_hand,2,0.0,0.0,0.0
471,20,20-left_hand-3,left_hand,3,0.0,0.0,0.0
472,20,20-left_hand-4,left_hand,4,0.0,0.0,0.0
473,20,20-left_hand-5,left_hand,5,0.0,0.0,0.0
474,20,20-left_hand-6,left_hand,6,0.0,0.0,0.0
475,20,20-left_hand-7,left_hand,7,0.0,0.0,0.0
476,20,20-left_hand-8,left_hand,8,0.0,0.0,0.0
477,20,20-left_hand-9,left_hand,9,0.0,0.0,0.0


In [9]:
import numpy as np

# 1. Extract just the raw coordinate numbers from Pandas
raw_coordinates = clean_hands_df[['x', 'y', 'z']].values

# 2. Reshape the array
# -1 tells NumPy to automatically figure out the number of frames based on the data length
# 126 is our calculated feature size (42 landmarks * 3 coordinates)
sequence_array = raw_coordinates.reshape(-1, 126)

print(f"Old Pandas shape: {raw_coordinates.shape}")
print(f"New Neural Network shape: {sequence_array.shape}")

Old Pandas shape: (966, 3)
New Neural Network shape: (23, 126)


# 3. Video Length Statistics & Profiling

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
from tqdm import tqdm

frame_counts = []

print("Scanning dataset for video lengths (this will take a minute or two)...")

# Loop through every file path in your master train.csv
for file_path in tqdm(df['path']):
    # Read ONLY the lightweight metadata, not the whole massive file
    metadata = pq.read_metadata('../data/raw/' + file_path)
    
    # Calculate total frames (Total rows / 543 landmarks per frame)
    num_frames = metadata.num_rows // 543
    frame_counts.append(num_frames)

# Convert our list to a Pandas Series to easily calculate the math
lengths = pd.Series(frame_counts)

print("\n--- Video Length Statistics ---")
print(f"Minimum frames: {lengths.min()}")
print(f"Maximum frames: {lengths.max()}")
print(f"Average frames: {lengths.mean():.2f}")
print(f"Median frames:  {lengths.median()}")

# 4. Data Preprocessing & Label Encoding

In [11]:
import numpy as np
import pandas as pd

MAX_FRAMES = 64
NUM_FEATURES = 126

def process_video_file(file_path):
    # 1. Load the data
    df = pd.read_parquet(file_path)
    
    # 2. Filter out face and pose, keeping only hands
    hands_df = df[df['type'].isin(['left_hand', 'right_hand'])]
    
    # 3. Clean missing data
    hands_df = hands_df.fillna(0.0)
    
    # 4. Reshape into (Frames, 126)
    coords = hands_df[['x', 'y', 'z']].values
    sequence = coords.reshape(-1, NUM_FEATURES)
    
    # 5. Pad or Truncate to exactly 64 frames
    current_frames = sequence.shape[0]
    if current_frames < MAX_FRAMES:
        padding = np.zeros((MAX_FRAMES - current_frames, NUM_FEATURES))
        sequence = np.vstack((sequence, padding))
    elif current_frames > MAX_FRAMES:
        sequence = sequence[:MAX_FRAMES, :]
        
    return sequence

In [12]:
# Get a sorted list of all unique signs in the dataset
unique_signs = sorted(df['sign'].unique())

# Create a dictionary mapping each sign to a number (0 to 249)
sign_to_id = {sign: i for i, sign in enumerate(unique_signs)}

# Add a new column to our master dataframe with the numeric ID
df['sign_id'] = df['sign'].map(sign_to_id)

# Save the total number of classes for our neural network output layer
NUM_CLASSES = len(unique_signs)

print(f"Total unique signs to learn: {NUM_CLASSES}")
df[['sign', 'sign_id']].head()

Total unique signs to learn: 250


,sign,sign_id
0,blow,25
1,wait,232
2,cloud,48
3,bird,23
4,owie,164


# 5. Model Architecture & Batch Data Generator

In [13]:
import tensorflow as tf
import math

class ASLDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size=32):
        self.dataframe = dataframe
        self.batch_size = batch_size

    def __len__(self):
        # Calculates how many batches make up one full epoch
        return math.ceil(len(self.dataframe) / self.batch_size)

    def __getitem__(self, idx):
        # 1. Grab a chunk of 32 rows from the dataframe
        batch_df = self.dataframe.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        # 2. Create empty numpy arrays to hold our processed batch
        X = np.zeros((len(batch_df), MAX_FRAMES, NUM_FEATURES))
        y = np.zeros((len(batch_df),))
        
        # 3. Process each file in the batch on-the-fly
        for i, (_, row) in enumerate(batch_df.iterrows()):
            # Use the process_video_file function we built earlier
            X[i] = process_video_file('../data/raw/' + row['path'])
            y[i] = row['sign_id']
            
        return X, y

# Initialize the generator
train_generator = ASLDataGenerator(df, batch_size=32)
print("Data Generator ready!")

Data Generator ready!


In [15]:
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout, BatchNormalization

model = Sequential([
    # Explicit Input layer instead of input_shape argument
    Input(shape=(MAX_FRAMES, NUM_FEATURES)),
    
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    BatchNormalization(),
    
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    BatchNormalization(),
    
    Dense(64, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

# 6. Training & Checkpointing

In [16]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint('asl_model_best.h5', monitor='accuracy', save_best_only=True)
]

print("Initiating memory-safe training sequence...")

# The fix: We restrict max_queue_size to prevent the RAM explosion
history = model.fit(
    train_generator,
    epochs=10,
    callbacks=callbacks,
    max_queue_size=2,  # Only pre-load 2 batches at a time
    workers=1,         # Force it to use a single data-loading thread
    use_multiprocessing=False
)

Initiating memory-safe training sequence...
Epoch 1/10
2953/2953 [==============================] - 930s 312ms/step - loss: 5.3131 - accuracy: 0.0121
Epoch 2/10
2953/2953 [==============================] - 949s 321ms/step - loss: 4.5171 - accuracy: 0.0630
Epoch 3/10
2953/2953 [==============================] - 996s 337ms/step - loss: 3.8270 - accuracy: 0.1428
Epoch 4/10
2953/2953 [==============================] - 909s 308ms/step - loss: 3.4022 - accuracy: 0.2142
Epoch 5/10
2953/2953 [==============================] - 915s 310ms/step - loss: 3.1100 - accuracy: 0.2704
Epoch 6/10
2953/2953 [==============================] - 917s 311ms/step - loss: 2.8876 - accuracy: 0.3137
Epoch 7/10
2953/2953 [==============================] - 919s 311ms/step - loss: 2.7034 - accuracy: 0.3514
Epoch 8/10
2953/2953 [==============================] - 924s 313ms/step - loss: 2.5719 - accuracy: 0.3795
Epoch 9/10
2953/2953 [==============================] - 940s 318ms/step - loss: 2.4647 - accuracy: 0.4031
Ep

# 7. Model Export

In [17]:
# Save the complete model (architecture + weights + optimizer state)
model.save('../models/asl_v1_raw_baseline.keras')
print("V1 Baseline successfully saved!")

V1 Baseline successfully saved!


# 8. Feature Engineering & Velocity Extraction Generator

In [ ]:
# ==========================================
# 1. THE FEATURE EXTRACTOR
# ==========================================
def extract_features(sequence_array):
    """
    Takes raw coordinates, calculates velocity, and doubles the feature count.
    """
    # Calculate velocity (deltas) between consecutive frames
    velocity = np.diff(sequence_array, axis=0)
    
    # Pad the first frame with zeros so the frame count doesn't shrink
    velocity = np.pad(velocity, ((1, 0), (0, 0)), mode='constant')
    
    # Combine the raw X,Y,Z data with the new Velocity data
    enhanced_features = np.concatenate([sequence_array, velocity], axis=-1)
    return enhanced_features

# ==========================================
# 2. THE V2 DATA GENERATOR
# ==========================================
class ASLDataGeneratorV2(tf.keras.utils.Sequence): # Renamed to V2
    def __init__(self, dataframe, batch_size=32):
        self.dataframe = dataframe
        self.batch_size = batch_size
        self.indices = np.arange(len(self.dataframe))
        
    def __len__(self):
        # Calculates how many batches per epoch
        return int(np.floor(len(self.dataframe) / self.batch_size))
    
    def __getitem__(self, index):
        # Grabs the specific chunk of the dataframe for this batch
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_df = self.dataframe.iloc[batch_indices]
        
        X_batch = []
        y_batch = []
        
        for _, row in batch_df.iterrows():
            file_path = row['path']
            
            # 1. Load ONLY needed columns (Fast I/O)
            raw_data = pd.read_parquet('../data/raw/' + file_path, columns=['type', 'x', 'y', 'z'])
            
            # 2. Filter for hands
            hands_df = raw_data[(raw_data['type'] == 'left_hand') | (raw_data['type'] == 'right_hand')]
            coords = hands_df[['x', 'y', 'z']].fillna(0).values 
            
            # 3. THE CRITICAL SHAPE FIX
            # Group every 42 landmarks (1 frame) into a single horizontal row of 126 features (42 * 3)
            num_frames = len(coords) // 42
            frames_array = coords.reshape(num_frames, 126)
            
            # 4. Inject V2 Math (Calculates velocity frame-to-frame)
            enhanced_array = extract_features(frames_array)
            
            X_batch.append(enhanced_array)
            # FIX: Use sign_id from the EDA section, not 'label'
            y_batch.append(row['sign_id']) 
            
        # Capping videos at 150 frames. Anything longer gets trimmed, anything shorter gets padded.
        X_batch_padded = pad_sequences(X_batch, maxlen=150, padding='post', truncating='post', dtype='float32')
        
        return X_batch_padded, np.array(y_batch)
        
# Initialize the V2 generator (FIX: using 'df' instead of 'train_df')
train_generator_v2 = ASLDataGeneratorV2(dataframe=df, batch_size=32)

# ==========================================
# 3. THE UPDATED ARCHITECTURE (V2)
# ==========================================
model_v2 = Sequential([ # Renamed to model_v2
    # Masking layer tells the LSTM to ignore the 0s we padded earlier
    Masking(mask_value=0.0, input_shape=(None, 252)),
    
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    # Dynamically use NUM_CLASSES from earlier in the notebook
    Dense(NUM_CLASSES, activation='softmax') 
])

model_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 4. THE MEMORY-SAFE TRAINING LOOP
# ==========================================
callbacks_v2 = [ # Renamed callbacks
    EarlyStopping(monitor='accuracy', patience=3, restore_best_weights=True),
    ModelCheckpoint('../models/asl_v2_engineered.keras', monitor='accuracy', save_best_only=True)
]

print("Initiating V2 Feature-Engineered Training Sequence...")
history_v2 = model_v2.fit(
    train_generator_v2, # Using the V2 generator
    epochs=10,
    callbacks=callbacks_v2,
    max_queue_size=10,  
    workers=1,         
    use_multiprocessing=False
)